# Resolução — S03-02

Notebook em **Julia**, corrigido para atender aos requisitos pedido pelo professor 
Prof. Kenny Vinente dos Santos, Dr. | kennyvinente@ufam.edu.br
# Nome: Renyer Montefusco levy
# Matrícula: 2240917



## Pacotes importados

In [1]:
using LinearAlgebra
using Printf

## Funções auxiliares para exibir histórico

As funções abaixo servem apenas para organizar as respostas e mostrar as iterações de cada método.


In [2]:
function mostrar_historico_1variavel(historico)
    println("Iteração | x | F(x)")
    println("---------------------------------------------")
    for linha in historico
        @printf("%8d | %.16f | %.16e\n", linha.Iteracao, linha.x, linha.Fx)
    end
end

function mostrar_historico_sistema(historico)
    println("Iteração | x | ||F(x)||")
    println("--------------------------------------------------------")
    for linha in historico
        @printf("%8d | [% .16f, % .16f] | %.16e\n",
            linha.Iteracao,
            linha.x[1],
            linha.x[2],
            linha.Norma
        )
    end
end

mostrar_historico_sistema (generic function with 1 method)

# Algorithm 8.1 — Finite Difference Newton's Method: One Variable

O método calcula uma aproximação da derivada usando diferença finita.

Para uma variável, usamos:

\[
s =
\begin{cases}
\tau x_k, & \text{se } |x_k| > 1 \\
\tau, & \text{caso contrário}
\end{cases}
\]

e:

\[
x_{k+1}=x_k-\frac{sF(x_k)}{F(x_k+s)-F(x_k)}
\]


In [3]:
function finite_difference_newton_1d(F, x0; tau=1e-7, eps=1e-12, maxiter=100)
    x = float(x0)
    historico = []

    for k in 0:maxiter
        Fx = F(x)
        push!(historico, (Iteracao=k, x=x, Fx=Fx))

        if abs(Fx) <= eps
            break
        end

        if abs(x) > 1
            s = tau*x
        else
            s = tau
        end

        denominador = F(x + s) - Fx

        if abs(denominador) < eps
            println("Denominador muito próximo de zero. Método interrompido.")
            break
        end

        x = x - (s*Fx)/denominador
    end

    return x, historico
end

finite_difference_newton_1d (generic function with 1 method)

## Exemplo 1

Resolver:

\[
F(x)=x^2-2
\]

com:

\[
x_0=2
\]

e:

\[
\tau=10^{-7}
\]


In [4]:
F(x) = x^2 - 2

raiz, historico = finite_difference_newton_1d(F, 2.0; tau=1e-7)

mostrar_historico_1variavel(historico)

println()
println("Raiz aproximada = ", raiz)
println("F(raiz) = ", F(raiz))
println("Erro absoluto = ", abs(F(raiz)))

Iteração | x | F(x)
---------------------------------------------
       0 | 2.0000000000000000 | 2.0000000000000000e+00
       1 | 1.5000000252719503 | 2.5000007581585137e-01
       2 | 1.4166666722399401 | 6.9444602353860851e-03
       3 | 1.4142156864080049 | 6.0076824643928717e-06
       4 | 1.4142135623747960 | 4.8108184103057283e-12
       5 | 1.4142135623730951 | 4.4408920985006262e-16

Raiz aproximada = 1.4142135623730951
F(raiz) = 4.440892098500626e-16
Erro absoluto = 4.440892098500626e-16


## Exemplo 2

Executar o mesmo método com:

\[
x_0=2
\]

e:

\[
\tau=0.1
\]


In [5]:
F(x) = x^2 - 2

raiz, historico = finite_difference_newton_1d(F, 2.0; tau=0.1)

mostrar_historico_1variavel(historico)

println()
println("Raiz aproximada = ", raiz)
println("F(raiz) = ", F(raiz))
println("Erro absoluto = ", abs(F(raiz)))

Iteração | x | F(x)
---------------------------------------------
       0 | 2.0000000000000000 | 2.0000000000000000e+00
       1 | 1.5238095238095242 | 3.2199546485260866e-01
       2 | 1.4231859410430840 | 2.5458222782688722e-02
       3 | 1.4146677545712822 | 1.2848558237537056e-03
       4 | 1.4142352600123498 | 6.1370662198800829e-05
       5 | 1.4142145957525318 | 2.9228394970992611e-06
       6 | 1.4142136115819992 | 1.3918380181721091e-07
       7 | 1.4142135647163769 | 6.6278018628906921e-09
       8 | 1.4142135624846799 | 3.1560976054834100e-10
       9 | 1.4142135623784085 | 1.5028422950535969e-11
      10 | 1.4142135623733481 | 7.1542771706845087e-13

Raiz aproximada = 1.414213562373348
F(raiz) = 7.154277170684509e-13
Erro absoluto = 7.154277170684509e-13


# Algorithm 8.2 — Secant Method: One Variable

O método da secante usa uma aproximação da derivada.

A atualização é:

\[
x_{k+1}=x_k-\frac{F(x_k)}{a_k}
\]

e a aproximação da derivada é atualizada por:

\[
a_{k+1}=\frac{F(x_k)-F(x_{k+1})}{x_k-x_{k+1}}
\]


In [6]:
function secant_1d(F, x0; a0=1.0, eps=1e-12, maxiter=100)
    x = float(x0)
    a = float(a0)
    historico = []

    for k in 0:maxiter
        Fx = F(x)
        push!(historico, (Iteracao=k, x=x, Fx=Fx, a=a))

        if abs(Fx) <= eps
            break
        end

        if abs(a) < eps
            println("Aproximação da derivada muito próxima de zero. Método interrompido.")
            break
        end

        x_novo = x - Fx/a
        Fx_novo = F(x_novo)

        denominador = x - x_novo

        if abs(denominador) < eps
            println("Denominador muito próximo de zero. Método interrompido.")
            break
        end

        a = (Fx - Fx_novo)/denominador
        x = x_novo
    end

    return x, historico
end

function mostrar_historico_secante_1d(historico)
    println("Iteração | x | F(x) | a")
    println("-----------------------------------------------------------")
    for linha in historico
        @printf("%8d | %.16f | %.16e | %.16f\n",
            linha.Iteracao,
            linha.x,
            linha.Fx,
            linha.a
        )
    end
end

mostrar_historico_secante_1d (generic function with 1 method)

## Exemplo 3

Resolver:

\[
F(x)=x^2-2
\]

com:

\[
x_0=2
\]

e:

\[
a_0=1
\]


In [7]:
F(x) = x^2 - 2

raiz, historico = secant_1d(F, 2.0; a0=1.0)

mostrar_historico_secante_1d(historico)

println()
println("Raiz aproximada = ", raiz)
println("F(raiz) = ", F(raiz))
println("Erro absoluto = ", abs(F(raiz)))

Iteração | x | F(x) | a
-----------------------------------------------------------
       0 | 2.0000000000000000 | 2.0000000000000000e+00 | 1.0000000000000000
       1 | 0.0000000000000000 | -2.0000000000000000e+00 | 2.0000000000000000
       2 | 1.0000000000000000 | -1.0000000000000000e+00 | 1.0000000000000000
       3 | 2.0000000000000000 | 2.0000000000000000e+00 | 3.0000000000000000
       4 | 1.3333333333333335 | -2.2222222222222188e-01 | 3.3333333333333335
       5 | 1.4000000000000001 | -3.9999999999999591e-02 | 2.7333333333333347
       6 | 1.4146341463414633 | 1.1897679952408424e-03 | 2.8146341463414699
       7 | 1.4142114384748701 | -6.0072868388605372e-06 | 2.8288455848168468
       8 | 1.4142135620573204 | -8.9314555751229818e-10 | 2.8284250005774818
       9 | 1.4142135623730954 | 8.8817841970012523e-16 | 2.8284270569936236

Raiz aproximada = 1.4142135623730954
F(raiz) = 8.881784197001252e-16
Erro absoluto = 8.881784197001252e-16


# Algorithm 8.3 — Finite Difference Newton's Method: n Variables

Para sistemas, montamos uma aproximação da matriz Jacobiana usando diferenças finitas.

Para cada variável \(j\), escolhemos:

\[
s_j =
\begin{cases}
\tau (x_k)_j, & \text{se } |(x_k)_j| \geq 1 \\
\tau, & \text{se } 0 \leq (x_k)_j < 1 \\
-\tau, & \text{se } -1 < (x_k)_j < 0
\end{cases}
\]

Depois calculamos a coluna \(j\) da matriz aproximada \(A_k\):

\[
(A_k)_j=\frac{F(x_k+s_je_j)-F(x_k)}{s_j}
\]

Por fim, resolvemos:

\[
A_k d_{k+1}=-F(x_k)
\]

e atualizamos:

\[
x_{k+1}=x_k+d_{k+1}
\]


In [8]:
function matriz_diferencas_finitas(F, x; tau=1e-7)
    n = length(x)
    A = zeros(n, n)
    Fx = F(x)

    for j in 1:n
        sj = 0.0

        if abs(x[j]) >= 1
            sj = tau*x[j]
        elseif 0 <= x[j] < 1
            sj = tau
        else
            sj = -tau
        end

        e = zeros(n)
        e[j] = 1.0

        A[:, j] = (F(x + sj*e) - Fx)/sj
    end

    return A
end

function finite_difference_newton_system(F, x0; tau=1e-7, eps=1e-12, maxiter=100)
    x = float.(x0)
    historico = []

    for k in 0:maxiter
        Fx = F(x)
        norma = norm(Fx)

        push!(historico, (Iteracao=k, x=copy(x), Norma=norma))

        if norma <= eps
            break
        end

        A = matriz_diferencas_finitas(F, x; tau=tau)
        d = A \ (-Fx)

        x = x + d
    end

    return x, historico
end

finite_difference_newton_system (generic function with 1 method)

## Exemplo 4

Resolver o sistema:

\[
F(x)=
\begin{bmatrix}
(x_1+1)^2+x_2^2-2 \\
e^{x_1}+x_2^3-2
\end{bmatrix}
\]

com:

\[
x_0=
\begin{bmatrix}
1 \\
1
\end{bmatrix}
\]

e:

\[
\tau=10^{-7}
\]


In [9]:
function Fsistema(x)
    x1 = x[1]
    x2 = x[2]

    return [
        (x1 + 1)^2 + x2^2 - 2,
        exp(x1) + x2^3 - 2
    ]
end

solucao, historico = finite_difference_newton_system(Fsistema, [1.0, 1.0]; tau=1e-7)

mostrar_historico_sistema(historico)

println()
println("Solução aproximada = ", solucao)
println("F(solução) = ", Fsistema(solucao))
println("Norma do erro = ", norm(Fsistema(solucao)))

Iteração | x | ||F(x)||
--------------------------------------------------------
       0 | [ 1.0000000000000000,  1.0000000000000000] | 3.4572376895453050e+00
       1 | [ 0.1523592284107350,  1.1952815792526243] | 1.1547087761244197e+00
       2 | [-0.0108376850008005,  1.0361111885988827] | 1.1404263173224197e-01
       3 | [-0.0008896678328822,  1.0015353225002093] | 3.9423456113930298e-03
       4 | [-0.0000013701692951,  1.0000029389238940] | 8.0806136493301997e-06
       5 | [-0.0000000000056881,  1.0000000000111586] | 2.9863924458983494e-11
       6 | [-0.0000000000000001,  1.0000000000000002] | 4.4408920985006262e-16

Solução aproximada = [-1.1072444444653868e-16, 1.0000000000000002]
F(solução) = [0.0, 4.440892098500626e-16]
Norma do erro = 4.440892098500626e-16


## Exemplo 5

Executar o mesmo método com:

\[
x_0=
\begin{bmatrix}
1 \\
1
\end{bmatrix}
\]

e:

\[
\tau=0.1
\]


In [10]:
solucao, historico = finite_difference_newton_system(Fsistema, [1.0, 1.0]; tau=0.1)

mostrar_historico_sistema(historico)

println()
println("Solução aproximada = ", solucao)
println("F(solução) = ", Fsistema(solucao))
println("Norma do erro = ", norm(Fsistema(solucao)))

Iteração | x | ||F(x)||
--------------------------------------------------------
       0 | [ 1.0000000000000000,  1.0000000000000000] | 3.4572376895453050e+00
       1 | [ 0.1646296592891310,  1.2023897128164591] | 1.2185277812214399e+00
       2 | [-0.0145741083381838,  1.0571349919803852] | 1.8897289752361113e-01
       3 | [-0.0057235630095866,  1.0097667847229959] | 2.5253622800040142e-02
       4 | [-0.0004768963601209,  1.0012201624741157] | 3.5184272507079615e-03
       5 | [-0.0000665078241251,  1.0001416711481008] | 3.8881423590779538e-04
       6 | [-0.0000070228456769,  1.0000162771563474] | 4.5723191163713869e-05
       7 | [-0.0000008386496296,  1.0000018684283873] | 5.1925631370241104e-06
       8 | [-0.0000000945112631,  1.0000002144203548] | 5.9886499987183426e-07
       9 | [-0.0000000109384631,  1.0000000246077563] | 6.8570382536612082e-08
      10 | [-0.0000000012504224,  1.0000000028240119] | 7.8775910138391665e-09
      11 | [-0.0000000001437612,  1.00000000032409

# Algorithm 8.4 — Secant Method: n Variables

Este método usa a atualização de Broyden para aproximar a Jacobiana.

Inicialmente:

\[
x_1=x_0-A_0^{-1}F(x_0)
\]

Depois:

\[
d_0=x_1-x_0
\]

\[
y_0=F(x_1)-F(x_0)
\]

A atualização de Broyden é:

\[
A_k=A_{k-1}+\frac{(y_{k-1}-A_{k-1}d_{k-1})d_{k-1}^T}{d_{k-1}^T d_{k-1}}
\]

Em seguida, resolvemos:

\[
A_kd_k=-F(x_k)
\]

e atualizamos:

\[
x_{k+1}=x_k+d_k
\]


In [11]:
function secant_system_broyden(F, x0; A0=nothing, eps=1e-12, maxiter=100)
    x0 = float.(x0)
    n = length(x0)

    if A0 === nothing
        A = Matrix{Float64}(I, n, n)
    else
        A = float.(A0)
    end

    F0 = F(x0)
    x1 = x0 - A \ F0

    d_ant = x1 - x0
    y_ant = F(x1) - F0

    x = x1
    historico = []

    push!(historico, (Iteracao=0, x=copy(x0), Norma=norm(F0)))
    push!(historico, (Iteracao=1, x=copy(x), Norma=norm(F(x))))

    for k in 1:maxiter
        Fx = F(x)

        if norm(Fx) <= eps
            break
        end

        denominador = dot(d_ant, d_ant)

        if abs(denominador) < eps
            println("Denominador muito próximo de zero. Método interrompido.")
            break
        end

        A = A + ((y_ant - A*d_ant) * transpose(d_ant)) / denominador

        d = A \ (-Fx)
        x_novo = x + d

        y = F(x_novo) - Fx

        x = x_novo
        d_ant = d
        y_ant = y

        push!(historico, (Iteracao=k+1, x=copy(x), Norma=norm(F(x))))

        if norm(F(x)) <= eps
            break
        end
    end

    return x, historico
end

secant_system_broyden (generic function with 1 method)

## Exemplo 6 — Example 7.11

Resolver:

\[
F(x)=
\begin{bmatrix}
(x_1+1)^2+x_2^2-2 \\
e^{x_1}+x_2^3-2
\end{bmatrix}
\]

com:

\[
x_0=
\begin{bmatrix}
1 \\
1
\end{bmatrix}
\]

Usando \(A_0=I\), conforme valor padrão do algoritmo.


In [12]:
solucao, historico = secant_system_broyden(Fsistema, [1.0, 1.0])

mostrar_historico_sistema(historico)

println()
println("Solução aproximada = ", solucao)
println("F(solução) = ", Fsistema(solucao))
println("Norma do erro = ", norm(Fsistema(solucao)))

Denominador muito próximo de zero. Método interrompido.
Iteração | x | ||F(x)||
--------------------------------------------------------
       0 | [ 1.0000000000000000,  1.0000000000000000] | 3.4572376895453050e+00
       1 | [-2.0000000000000000, -0.7182818284590451] | 2.2870623161399628e+00
       2 | [-1.6645002598330279,  0.8309215956438640] | 1.5111783668476306e+00
       3 | [-0.2425625640493636,  2.0377121350694134] | 7.7415651325325037e+00
       4 | [-1.2415558219417240,  0.7712913290252346] | 1.8389803016557322e+00
       5 | [-0.5805216684286952,  0.4222117814286767] | 2.1382593321570686e+00
       6 | [-3.2619320912082208,  2.0382595966574515] | 9.7568758203831081e+00
       7 | [-1.1090527426757810,  0.7075096120401092] | 1.9860884230371261e+00
       8 | [-1.4790623218085062,  0.9313900788989945] | 1.3210119220455909e+00
       9 | [-1.8328702354887900,  1.2782832855438411] | 4.1135888735399029e-01
      10 | [-1.7071565995299662,  1.1909193465171115] | 1.531299894120806